# Noise-Robust Customer Support Intent Classification
### Final NLP Project — End-to-End Industry Track Solution (AWS SageMaker Ready)

---

> **How to Run on AWS SageMaker:**
> 1. Upload this single `.ipynb` file to **AWS SageMaker Studio** or a **SageMaker Notebook Instance** (recommended instance: `ml.g4dn.xlarge` or `ml.g5.xlarge` with a PyTorch GPU kernel; CPU fallback is also supported).
> 2. Click **Run → Run All Cells**. No external files, credentials, or manual downloads are required.
> 3. **Crash-Safe Checkpointing:** Every BiLSTM and DistilBERT model automatically saves an atomic checkpoint to `./outputs/checkpoints/` **after every epoch**. If SageMaker crashes, times out, or restarts, simply click **Run → Run All Cells** again — completed models are reloaded instantly and interrupted models resume from the next epoch!
> 4. All datasets, models, evaluation tables, confusion matrices, plots, failure analyses, and an interactive presentation dashboard are generated automatically and saved under `./outputs/`.

---

## 1. Milestone & Rubric Coverage Map (M1 – M6)

| Milestone | Industry Track Deliverable | Where Covered in This Notebook |
|---|---|---|
| **M1 — Problem Definition & Business Case** | Problem brief, users, business objective, success metrics, feasibility | Section 1 (below) |
| **M2 — Data Understanding & Preparation** | Data card, EDA, cleaning, statistical analysis, leak-free splits, privacy | Sections 2 – 3 |
| **M3 — Baseline & Model Development** | Traditional ML (LR, SVM), BiLSTM + Attention, DistilBERT (Clean & Robust), Experiment Log | Sections 4 – 7 |
| **M4 — Evaluation & Optimization** | 5-noise robustness benchmark, Phase 1 vs Phase 2 ablation, severity sweep, confusion matrices, calibration, failure analysis | Sections 8 – 10 |
| **M5 — Deployment Prototype** | Interactive live dashboard (`ipywidgets` + optional Gradio), latency benchmarks, confidence fallback, monitoring plan, S3 export | Sections 11 – 12 |
| **M6 — Final Delivery** | Executive summary, tested queries showcase, presentation figures/CSVs | Section 13 & `./outputs/` |

---

## M1 — Problem Definition & Business Case

### Problem Brief
Automated customer support chatbots and ticket-routing systems rely on **Intent Classification** to route incoming user queries to the right workflow (e.g., `cancel_order`, `get_refund`, `track_order`, `reset_password`). While pretrained transformers achieve >98% accuracy on clean, well-formed queries, real-world customer messages contain **typos, SMS abbreviations, slang, grammatical errors, and indirect phrasing**. Under such noise, naive classifiers degrade sharply or make **high-confidence routing errors**, causing frustrated customers and expensive manual escalations.

### Target Users & Business Objective
- **Primary Users:** Customer Support Operations teams, Conversational AI engineers, and end-customers interacting with live support chat.
- **Business Objective:** Build a **noise-robust, calibrated 27-class intent classification system** that maintains high accuracy across all five real-world noise types while reliably flagging uncertain predictions for human-agent handoff.

### Measurable Success Metrics
1. **Clean Accuracy & Macro-F1:** ≥ 97% on clean held-out customer support queries.
2. **Noise Robustness Retention:** Maintain ≥ 90% accuracy across individual noise types and significantly outperform clean-trained baselines on combined multi-noise stress tests.
3. **Calibration & Safe Escalation:** Reduce Expected Calibration Error (ECE) via temperature scaling so low-confidence thresholding catches errors before they reach customers.
4. **Production Latency:** < 25 ms/query on GPU (and < 80 ms/query on CPU) suitable for real-time chat routing.

### Feasibility & Evolution (Phase 1 → Phase 2)
- **Phase 1 (Baseline Transformer):** Fine-tuned DistilBERT on the 27-class Bitext dataset with basic spelling/abbreviation augmentation and a basic prediction interface.
- **Phase 2 (Final System — This Notebook):** Expands to **all 5 noise types** (typos, abbreviations, slang, grammatical errors, indirect language) + **combined multi-noise**, introduces an improved **mixed-severity noise-augmentation + label-smoothing + temperature-calibration** strategy, systematically compares **Traditional ML (Logistic Regression, Linear SVM)**, **BiLSTM with Attention**, **DistilBERT-Clean**, and **DistilBERT-Robust**, runs an **ablation & severity sweep**, performs **categorized failure analysis**, and delivers an **interactive live dashboard**.

## 2. Environment Setup & Reproducibility Configuration

This cell installs any missing packages (without overwriting SageMaker's CUDA-matched PyTorch), detects available hardware (`CUDA` > `MPS` > `CPU`), and centralizes all hyperparameters in `CONFIG`.

> **Runtime Tip:** With `FAST_DEMO_MODE = False` (default), full training + evaluation + ablation runs in **~10–15 minutes on a SageMaker `ml.g4dn.xlarge` (T4 GPU)**. If you are testing on a CPU-only machine, set `FAST_DEMO_MODE = True` for a quick 2–3 minute end-to-end verification run.

In [ ]:
# Install required packages cleanly without touching SageMaker's preinstalled CUDA PyTorch
%pip install -q transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn tqdm ipywidgets

In [ ]:
import os
import re
import gc
import csv
import sys
import json
import math
import time
import random
import urllib.request
import ssl
import warnings
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 120

# ═══════════════════════════════════════════════════════════════════════════════
# Central Configuration
# ═══════════════════════════════════════════════════════════════════════════════
FAST_DEMO_MODE = False  # Set True ONLY for quick CPU smoke-testing

CONFIG = {
    "seed": 42,
    "dataset_hf_id": "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    "dataset_fallback_url": (
        "https://huggingface.co/datasets/bitext/"
        "Bitext-customer-support-llm-chatbot-training-dataset/resolve/main/"
        "Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv"
    ),
    "output_dir": "outputs",
    # Split ratios (leak-free stratified)
    "val_size": 0.15,
    "test_size": 0.15,
    # Noise augmentation settings (Phase 2)
    "aug_copies_per_clean": 1 if FAST_DEMO_MODE else 2,  # 2 noisy copies per clean train utterance
    "max_train_samples": 4000 if FAST_DEMO_MODE else None,  # None = use full dataset (~26.8K)
    "max_eval_samples": 1000 if FAST_DEMO_MODE else None,
    # DistilBERT settings
    "transformer_model": "distilbert-base-uncased",
    "max_seq_length": 96,
    "bert_batch_size": 32,
    "bert_epochs": 2 if FAST_DEMO_MODE else 4,
    "bert_lr": 3e-5,
    "bert_weight_decay": 0.01,
    "bert_warmup_ratio": 0.1,
    "label_smoothing": 0.05,
    # BiLSTM settings
    "bilstm_vocab_size": 15000,
    "bilstm_embed_dim": 128,
    "bilstm_hidden_dim": 128,
    "bilstm_num_layers": 2,
    "bilstm_dropout": 0.35,
    "bilstm_batch_size": 64,
    "bilstm_epochs": 4 if FAST_DEMO_MODE else 10,
    "bilstm_lr": 1.5e-3,
    # Escalation threshold
    "default_confidence_threshold": 0.75,
    # Crash-safe checkpoint resumption (saves after every epoch)
    "resume_from_checkpoints": True,
}


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    if torch.cuda.is_available():
        dev = torch.device("cuda")
        gpu_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"[DEVICE] CUDA GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        dev = torch.device("mps")
        print("[DEVICE] Apple Silicon MPS")
    else:
        dev = torch.device("cpu")
        print("[DEVICE] CPU detected. Training DistilBERT on CPU will be slow; "
              "use ml.g4dn.xlarge on SageMaker or set FAST_DEMO_MODE = True.")
    return dev


set_seed(CONFIG["seed"])
DEVICE = get_device()
USE_AMP = (DEVICE.type == "cuda")

# Create output directories
OUT_DIR = Path(CONFIG["output_dir"])
for sub in ["models", "checkpoints", "plots", "tables", "reports"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)


def save_checkpoint_atomic(state: dict, ckpt_path: Path):
    """Atomically save checkpoint so a mid-write crash never corrupts the last valid file."""
    tmp_path = ckpt_path.with_suffix(".pt.tmp")
    torch.save(state, tmp_path)
    os.replace(tmp_path, ckpt_path)


def load_checkpoint_safe(ckpt_path: Path):
    """Load a PyTorch checkpoint safely across PyTorch 1.13–2.6+ versions."""
    try:
        return torch.load(ckpt_path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(ckpt_path, map_location="cpu")


print(f"[CONFIG] FAST_DEMO_MODE={FAST_DEMO_MODE} | Mixed Precision (AMP)={USE_AMP} | Auto-Resume={CONFIG['resume_from_checkpoints']}")
print(f"[CONFIG] Artifacts & Checkpoints will be saved to: {OUT_DIR.resolve()}")

## 3. M2 — Data Understanding, Cleaning, EDA & Privacy

### Data Card — Bitext Customer Support Dataset
- **Source:** `bitext/Bitext-customer-support-llm-chatbot-training-dataset` (HuggingFace Hub)
- **Domain:** E-commerce & SaaS customer support inquiries
- **Scale:** ~26,872 utterances across **27 intent classes** grouped into **11 high-level categories** (e.g., `ORDER`, `REFUND`, `PAYMENT`, `ACCOUNT`, `SHIPPING`, `FEEDBACK`).
- **Privacy & PII Handling:** The dataset replaces personally identifiable information (PII) with explicit template placeholders such as `{{Order Number}}`, `{{Account Type}}`, `{{Person Name}}`, and `{{Delivery City}}`. No real customer PII is stored or exposed, complying with GDPR/privacy guidelines. Our preprocessing preserves these structural tokens cleanly and our noise engine avoids corrupting `{{...}}` placeholders.
- **Leakage Prevention:** We remove exact duplicates *before* performing a stratified **70% Train / 15% Validation / 15% Test** split and verify **0% overlap** between splits. All synthetic noise augmentation is applied **strictly inside the training split**.

In [ ]:
def load_bitext_dataset():
    """Load Bitext 27-intent customer support dataset with automatic fallback."""
    df = None
    try:
        from datasets import load_dataset
        print("[DATA] Loading Bitext dataset from HuggingFace Hub...")
        ds = load_dataset(CONFIG["dataset_hf_id"], split="train")
        df = ds.to_pandas()
        print(f"[DATA] Loaded {len(df):,} rows via HuggingFace datasets.")
    except Exception as e:
        print(f"[WARN] HuggingFace datasets load failed ({e}). Trying direct CSV download...")
        csv_path = OUT_DIR / "bitext_raw.csv"
        if not csv_path.exists():
            ctx = ssl._create_unverified_context()
            req = urllib.request.Request(
                CONFIG["dataset_fallback_url"],
                headers={"User-Agent": "Mozilla/5.0"},
            )
            with urllib.request.urlopen(req, context=ctx, timeout=60) as resp, open(csv_path, "wb") as f:
                f.write(resp.read())
        df = pd.read_csv(csv_path)
        print(f"[DATA] Loaded {len(df):,} rows via direct CSV fallback.")

    # Standardize column names
    rename_map = {}
    for col in df.columns:
        cl = col.strip().lower()
        if cl in ("instruction", "utterance", "text", "query"):
            rename_map[col] = "text"
        elif cl == "intent":
            rename_map[col] = "intent"
        elif cl == "category":
            rename_map[col] = "category"
        elif cl == "flags":
            rename_map[col] = "flags"
    df = df.rename(columns=rename_map)
    assert "text" in df.columns and "intent" in df.columns, f"Unexpected columns: {df.columns.tolist()}"
    return df


raw_df = load_bitext_dataset()
print("\nColumns:", raw_df.columns.tolist())
display(raw_df[["text", "intent", "category"]].head(5))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Data Cleaning, Statistical Analysis & Leak-Free Stratified Splits
# ═══════════════════════════════════════════════════════════════════════════════
def clean_text_basic(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

df = raw_df.copy()
n_raw = len(df)
n_null = df[["text", "intent"]].isna().any(axis=1).sum()
df = df.dropna(subset=["text", "intent"]).copy()
df["text"] = df["text"].apply(clean_text_basic)
df["intent"] = df["intent"].astype(str).str.strip()
if "category" not in df.columns:
    df["category"] = "GENERAL"
else:
    df["category"] = df["category"].astype(str).str.strip()

# Remove empty utterances and exact text duplicates (prevents train/test leakage!)
df = df[df["text"].str.len() > 0].copy()
df["text_norm_key"] = df["text"].str.lower().str.strip()
n_before_dedup = len(df)
df = df.drop_duplicates(subset=["text_norm_key"]).drop(columns=["text_norm_key"]).reset_index(drop=True)
n_dedup_removed = n_before_dedup - len(df)

# Map intents <-> IDs
LABEL_NAMES = sorted(df["intent"].unique().tolist())
NUM_CLASSES = len(LABEL_NAMES)
LABEL2ID = {lbl: idx for idx, lbl in enumerate(LABEL_NAMES)}
ID2LABEL = {idx: lbl for idx, lbl in enumerate(LABEL_NAMES)}
INTENT_TO_CATEGORY = df.groupby("intent")["category"].agg(lambda x: x.value_counts().index[0]).to_dict()

df["label"] = df["intent"].map(LABEL2ID)
df["word_count"] = df["text"].str.split().str.len()
df["char_count"] = df["text"].str.len()
df["has_pii_placeholder"] = df["text"].str.contains(r"\{\{.*?\}\}", regex=True)

print("═" * 65)
print("DATA CLEANING & PRIVACY AUDIT")
print("═" * 65)
print(f"  Raw rows:                    {n_raw:,}")
print(f"  Null rows dropped:           {n_null:,}")
print(f"  Exact duplicates removed:    {n_dedup_removed:,}")
print(f"  Clean unique utterances:     {len(df):,}")
print(f"  Number of intent classes:    {NUM_CLASSES}")
print(f"  Number of categories:        {df['category'].nunique()}")
print(f"  Word count (mean / med / p95): "
      f"{df['word_count'].mean():.1f} / {df['word_count'].median():.0f} / {df['word_count'].quantile(0.95):.0f}")
print(f"  Utterances w/ PII mask {{{{...}}}}: {df['has_pii_placeholder'].sum():,} "
      f"({df['has_pii_placeholder'].mean()*100:.1f}%) — Privacy preserved!")

# Stratified 70 / 15 / 15 split
train_df, temp_df = train_test_split(
    df,
    test_size=(CONFIG["val_size"] + CONFIG["test_size"]),
    stratify=df["label"],
    random_state=CONFIG["seed"],
)
val_ratio_of_temp = CONFIG["val_size"] / (CONFIG["val_size"] + CONFIG["test_size"])
val_df, test_df = train_test_split(
    temp_df,
    test_size=(1.0 - val_ratio_of_temp),
    stratify=temp_df["label"],
    random_state=CONFIG["seed"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Optional subsampling for FAST_DEMO_MODE
if CONFIG["max_train_samples"] and len(train_df) > CONFIG["max_train_samples"]:
    train_df, _ = train_test_split(
        train_df, train_size=CONFIG["max_train_samples"], stratify=train_df["label"], random_state=CONFIG["seed"]
    )
    train_df = train_df.reset_index(drop=True)
if CONFIG["max_eval_samples"] and len(val_df) > CONFIG["max_eval_samples"]:
    val_df, _ = train_test_split(
        val_df, train_size=CONFIG["max_eval_samples"], stratify=val_df["label"], random_state=CONFIG["seed"]
    )
    val_df = val_df.reset_index(drop=True)
if CONFIG["max_eval_samples"] and len(test_df) > CONFIG["max_eval_samples"]:
    test_df, _ = train_test_split(
        test_df, train_size=CONFIG["max_eval_samples"], stratify=test_df["label"], random_state=CONFIG["seed"]
    )
    test_df = test_df.reset_index(drop=True)

# Leakage assertion
train_set = set(train_df["text"].str.lower())
val_set = set(val_df["text"].str.lower())
test_set = set(test_df["text"].str.lower())
assert len(train_set & val_set) == 0, "Data leakage between train and val!"
assert len(train_set & test_set) == 0, "Data leakage between train and test!"
assert len(val_set & test_set) == 0, "Data leakage between val and test!"

print(f"\n  Split sizes -> Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print("  Leakage check: PASSED (0 exact-text overlaps across Train/Val/Test)")

# Save label mapping
with open(OUT_DIR / "models" / "label_mapping.json", "w") as f:
    json.dump({"label2id": LABEL2ID, "id2label": ID2LABEL, "intent_to_category": INTENT_TO_CATEGORY}, f, indent=2)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Exploratory Data Analysis (EDA) Visualizations
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Intent distribution (27 classes)
intent_counts = df["intent"].value_counts()
sns.barplot(y=intent_counts.index, x=intent_counts.values, ax=axes[0], palette="viridis")
axes[0].set_title(f"Utterance Count per Intent ({NUM_CLASSES} Classes)", fontweight="bold")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Intent")
axes[0].tick_params(axis="y", labelsize=8)

# 2. High-level Category distribution
cat_counts = df["category"].value_counts()
sns.barplot(y=cat_counts.index, x=cat_counts.values, ax=axes[1], palette="mako")
axes[1].set_title(f"High-Level Support Categories ({len(cat_counts)})", fontweight="bold")
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Category")

# 3. Utterance length distribution
sns.histplot(df["word_count"], bins=25, kde=True, ax=axes[2], color="#2b7bba")
axes[2].axvline(df["word_count"].mean(), color="red", linestyle="--", label=f"Mean: {df['word_count'].mean():.1f}")
axes[2].axvline(df["word_count"].quantile(0.95), color="orange", linestyle=":", label=f"95th pct: {df['word_count'].quantile(0.95):.0f}")
axes[2].set_title("Utterance Word Length Distribution", fontweight="bold")
axes[2].set_xlabel("Word Count")
axes[2].set_ylabel("Frequency")
axes[2].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / "plots" / "01_eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Phase 2 Noise Injection Engine (All 5 Real-World Noise Types + Combined)

To rigorously test and train for real-world customer support robustness, we implement a deterministic, linguistically grounded **5-Type Noise Engine** that preserves PII placeholders (`{{...}}`) while simulating authentic user behavior:

1. **Typos (`typos`)**: QWERTY keyboard-neighbor substitutions, adjacent character swaps, character deletions, and double-key presses.
2. **Abbreviations (`abbreviations`)**: 80+ real customer-support & SMS shorthands (`please → pls`, `account → acct`, `order → ord`, `delivery → deliv`, `refund → rfnd`, `password → pwd`, `you → u`, `thanks → thx`) plus consonant-skeleton compression when no dictionary match exists.
3. **Slang (`slang`)**: Informal chat markers (`yo`, `bro`, `tbh`, `ngl`, `fr`, `rn`, `asap`, `smh`) and colloquial substitutions (`want to → wanna`, `going to → gonna`, `problem → mess`, `money → cash`).
4. **Grammatical Errors (`grammar`)**: Article deletion (`a/an/the`), auxiliary/copula omission (`am/is/are/do/does`), preposition drops, verb inflection errors (`doesn't → don't`, `cancelled → cancel`), punctuation stripping, and irregular casing.
5. **Indirect Language (`indirect`)**: Hedged, verbose, conversational wrappers (*"Hey, not sure if this is the right place, sorry to bother you, but..."*) that dilute direct intent keywords and mimic polite or uncertain customers.
6. **Combined Multi-Noise (`combined`)**: Chains 2–3 noise types simultaneously to simulate worst-case messy live-chat messages.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Five Real-World Noise Generators + Combined Stress Test
# ═══════════════════════════════════════════════════════════════════════════════
QWERTY_NEIGHBORS = {
    "a": "qwsz", "b": "vghn", "c": "xdfv", "d": "erfcxs", "e": "wsdr",
    "f": "rtgvcd", "g": "tyhbvf", "h": "yujnbg", "i": "ujko", "j": "uikmnh",
    "k": "iolmj", "l": "opk", "m": "njk", "n": "bhjm", "o": "iklp",
    "p": "ol", "q": "wa", "r": "edft", "s": "wedxza", "t": "rfgy",
    "u": "yhji", "v": "cfgb", "w": "qase", "x": "zsdc", "y": "tghu", "z": "asx",
}

ABBREV_MAP = {
    "please": "pls", "you": "u", "your": "ur", "you're": "ur", "are": "r",
    "thanks": "thx", "thank": "thx", "because": "bc", "before": "b4",
    "tomorrow": "tmrw", "today": "2day", "tonight": "2nite", "about": "abt",
    "account": "acct", "order": "ord", "orders": "ords", "delivery": "deliv",
    "deliver": "deliv", "cancel": "cncl", "cancellation": "cncl", "cancelled": "cncld",
    "canceled": "cncld", "refund": "rfnd", "payment": "pymt", "payments": "pymts",
    "address": "addr", "password": "pwd", "information": "info", "customer": "cust",
    "service": "svc", "support": "supp", "number": "num", "message": "msg",
    "messages": "msgs", "confirm": "cnfrm", "confirmation": "cnfrm", "receive": "rcv",
    "received": "rcvd", "available": "avail", "without": "w/o", "with": "w/",
    "between": "btw", "something": "smth", "someone": "sm1", "people": "ppl",
    "problem": "prob", "problems": "probs", "package": "pkg", "shipping": "ship",
    "invoice": "inv", "invoices": "invs", "subscription": "sub", "newsletter": "nwsltr",
    "change": "chg", "update": "upd", "delete": "del", "check": "chk",
    "track": "trk", "tracking": "trkg", "contact": "cntct", "hours": "hrs",
    "minutes": "mins", "seconds": "secs", "immediately": "asap", "right now": "rn",
    "do not": "dont", "cannot": "cant", "could not": "couldnt", "would not": "wouldnt",
    "want to": "wanna", "going to": "gonna", "need to": "need 2", "to": "2",
    "for": "4", "through": "thru", "though": "tho", "probably": "prob",
    "definitely": "def", "feedback": "fdbk", "complaint": "cmplnt", "agent": "agnt",
    "human": "hmn", "review": "rev", "status": "stat", "return": "rtrn",
}

SLANG_PHRASES = [
    (r"\bwant to\b", "wanna"),
    (r"\bgoing to\b", "gonna"),
    (r"\bhave to\b", "gotta"),
    (r"\bgot to\b", "gotta"),
    (r"\bkind of\b", "kinda"),
    (r"\bsort of\b", "sorta"),
    (r"\bi don't know\b", "idk"),
    (r"\bi do not know\b", "idk"),
    (r"\bright now\b", "rn"),
    (r"\bas soon as possible\b", "asap"),
    (r"\bvery\b", "super"),
    (r"\breally\b", "lowkey"),
    (r"\bproblem\b", "mess"),
    (r"\bissue\b", "headache"),
    (r"\bmoney\b", "cash"),
    (r"\bhelp me\b", "hook me up"),
    (r"\bcompany\b", "u guys"),
    (r"\btalk to\b", "chat w"),
    (r"\bspeak to\b", "hit up"),
    (r"\bhello\b", "yo"),
    (r"\bhi\b", "yo"),
]

SLANG_PREFIXES = [
    "yo bro ", "hey fam ", "tbh ", "ngl ", "bruh ", "yo ", "ayy ", "lowkey ",
]
SLANG_SUFFIXES = [
    " fr", " rn", " asap", " tbh", " ngl", " smh", " pls bro", " lol",
]

GRAMMAR_REPLACEMENTS = [
    (r"\bdoesn't\b", "dont"),
    (r"\bdoes not\b", "no"),
    (r"\bdidn't\b", "didnt"),
    (r"\bcan't\b", "cant"),
    (r"\bi'm\b", "me"),
    (r"\bi have\b", "i got"),
    (r"\bhas been\b", "been"),
    (r"\bwas\b", "is"),
    (r"\bwere\b", "was"),
    (r"\bordered\b", "order"),
    (r"\bcancelled\b", "cancel"),
    (r"\bcanceled\b", "cancel"),
    (r"\bchanged\b", "change"),
    (r"\bneed\b", "needing"),
    (r"\bwant\b", "wanting"),
]

INDIRECT_TEMPLATES_LOW = [
    "Hey, quick question if you don't mind — {core}",
    "Hi there, I was just wondering about something: {core}",
    "Sorry to bother you, just wanted to check: {core}",
    "Not sure if you can help, but {core}",
]

INDIRECT_TEMPLATES_MED = [
    "Hey, I'm not totally sure if this is the right department, sorry to bother you, but basically {core}",
    "So I was talking to a friend and they told me I should reach out here since {core}",
    "Honestly I'm a little confused about how your site works, is there any chance {core}",
    "Hope you're having a good day! I've been meaning to look into this for a while — {core}",
    "Quick favor if someone happens to be free, I didn't see an obvious button so {core}",
]

INDIRECT_TEMPLATES_HIGH = [
    "Hey there, hope all is well! I don't normally message support and maybe I missed it in the FAQ somewhere, but my cousin mentioned you folks are pretty helpful, anyway the situation is that {core} — would really appreciate any pointer!",
    "Sorry for the long message, I've been clicking around the app for like ten minutes and honestly got lost between the menus, so I figured I'd just ask directly here if there's any way {core}, thanks in advance!",
    "Not trying to be difficult at all, I just wasn't sure what the normal policy is on your end because every store does it differently, so I was hoping someone could clarify {core} whenever you get a moment.",
]

PLACEHOLDER_RE = re.compile(r"(\{\{.*?\}\})")


def _protect_placeholders(text: str):
    """Split text into (token, is_placeholder) segments so {{...}} is never corrupted."""
    parts = PLACEHOLDER_RE.split(text)
    return [(p, bool(PLACEHOLDER_RE.fullmatch(p))) for p in parts if p]


def _corrupt_word_typo(word: str, rng: random.Random) -> str:
    if len(word) <= 3 or not word.isalpha():
        return word
    op = rng.choice(["swap", "sub", "del", "dup"])
    chars = list(word)
    idx = rng.randint(1, len(chars) - 2)
    if op == "swap":
        chars[idx], chars[idx + 1] = chars[idx + 1], chars[idx]
    elif op == "sub":
        c = chars[idx].lower()
        if c in QWERTY_NEIGHBORS:
            rep = rng.choice(QWERTY_NEIGHBORS[c])
            chars[idx] = rep.upper() if chars[idx].isupper() else rep
    elif op == "del":
        chars.pop(idx)
    elif op == "dup":
        chars.insert(idx, chars[idx])
    return "".join(chars)


def inject_typos(text: str, severity: str = "medium", rng: random.Random = None) -> str:
    rng = rng or random.Random()
    prob = {"low": 0.18, "medium": 0.35, "high": 0.55}[severity]
    out_parts = []
    changed = False
    for seg, is_ph in _protect_placeholders(text):
        if is_ph:
            out_parts.append(seg)
            continue
        words = seg.split(" ")
        new_words = []
        for w in words:
            if len(w) > 3 and w.isalpha() and rng.random() < prob:
                nw = _corrupt_word_typo(w, rng)
                if nw != w:
                    changed = True
                new_words.append(nw)
            else:
                new_words.append(w)
        out_parts.append(" ".join(new_words))
    result = "".join(out_parts)
    # Guarantee at least one typo on eligible words
    if not changed:
        words = result.split(" ")
        eligible = [i for i, w in enumerate(words) if len(w) > 3 and w.isalpha() and "{{" not in w]
        if eligible:
            i = rng.choice(eligible)
            words[i] = _corrupt_word_typo(words[i], rng)
            result = " ".join(words)
    return result


def inject_abbreviations(text: str, severity: str = "medium", rng: random.Random = None) -> str:
    rng = rng or random.Random()
    prob = {"low": 0.50, "medium": 0.80, "high": 1.00}[severity]
    out_parts = []
    changed = False
    for seg, is_ph in _protect_placeholders(text):
        if is_ph:
            out_parts.append(seg)
            continue
        words = seg.split(" ")
        new_words = []
        for w in words:
            clean_w = re.sub(r"[^a-zA-Z']", "", w).lower()
            if clean_w in ABBREV_MAP and rng.random() < prob:
                abbr = ABBREV_MAP[clean_w]
                # Preserve trailing punctuation
                m = re.search(r"[^a-zA-Z']+$", w)
                suffix = m.group(0) if m else ""
                new_words.append(abbr + suffix)
                changed = True
            else:
                new_words.append(w)
        out_parts.append(" ".join(new_words))
    result = "".join(out_parts)
    # Fallback consonant compression for longer words if nothing matched
    if not changed or severity == "high":
        words = result.split(" ")
        eligible = [i for i, w in enumerate(words) if len(w) >= 6 and w.isalpha() and "{{" not in w]
        if eligible:
            k = 1 if severity != "high" else min(2, len(eligible))
            for i in rng.sample(eligible, k):
                w = words[i]
                shortened = w[0] + re.sub(r"[aeiouAEIOU]", "", w[1:-1]) + w[-1]
                if len(shortened) >= 3:
                    words[i] = shortened
        result = " ".join(words)
    return result


def inject_slang(text: str, severity: str = "medium", rng: random.Random = None) -> str:
    rng = rng or random.Random()
    out_parts = []
    for seg, is_ph in _protect_placeholders(text):
        if is_ph:
            out_parts.append(seg)
            continue
        s = seg
        for pattern, repl in SLANG_PHRASES:
            if rng.random() < (0.6 if severity == "low" else 0.9):
                s = re.sub(pattern, repl, s, flags=re.IGNORECASE)
        out_parts.append(s)
    result = "".join(out_parts).strip()
    if severity == "low":
        if rng.random() < 0.7:
            result = rng.choice(SLANG_PREFIXES) + result
        else:
            result = result.rstrip(".!?") + rng.choice(SLANG_SUFFIXES)
    else:
        result = rng.choice(SLANG_PREFIXES) + result.rstrip(".!?") + rng.choice(SLANG_SUFFIXES)
    return re.sub(r"\s+", " ", result).strip()


def inject_grammatical_errors(text: str, severity: str = "medium", rng: random.Random = None) -> str:
    rng = rng or random.Random()
    drop_words = {"a", "an", "the", "am", "is", "are", "do", "does", "to", "for", "of", "would", "could"}
    drop_prob = {"low": 0.45, "medium": 0.75, "high": 0.95}[severity]
    out_parts = []
    for seg, is_ph in _protect_placeholders(text):
        if is_ph:
            out_parts.append(seg)
            continue
        s = seg
        for pat, repl in GRAMMAR_REPLACEMENTS:
            if rng.random() < drop_prob:
                s = re.sub(pat, repl, s, flags=re.IGNORECASE)
        words = s.split(" ")
        kept = []
        for w in words:
            lw = re.sub(r"[^a-zA-Z]", "", w).lower()
            if lw in drop_words and rng.random() < drop_prob and len(words) > 3:
                continue
            kept.append(w)
        s = " ".join(kept) if kept else s
        # Strip punctuation
        s = re.sub(r"[.,?!;:]", "", s)
        out_parts.append(s)
    result = re.sub(r"\s+", " ", "".join(out_parts)).strip()
    if severity == "high" and rng.random() < 0.35:
        result = result.upper()
    else:
        result = result.lower()
    return result


def inject_indirect_language(text: str, severity: str = "medium", rng: random.Random = None) -> str:
    rng = rng or random.Random()
    # Soften direct imperative openings so intent is more implicit
    core = text.strip()
    core = re.sub(r"^(please\s+|hi[,!]?\s+|hello[,!]?\s+|hey[,!]?\s+)", "", core, flags=re.IGNORECASE)
    if core and core[0].isupper() and not core.startswith("I ") and not core.startswith("{{"):
        core = core[0].lower() + core[1:]
    core = core.rstrip(".!?")
    templates = {
        "low": INDIRECT_TEMPLATES_LOW,
        "medium": INDIRECT_TEMPLATES_MED,
        "high": INDIRECT_TEMPLATES_HIGH,
    }[severity]
    return rng.choice(templates).format(core=core)


def inject_combined_noise(text: str, severity: str = "medium", rng: random.Random = None) -> str:
    rng = rng or random.Random()
    # Combine 2 or 3 distinct noise types in a realistic order
    k = 2 if severity == "low" else 3
    chosen = rng.sample(["indirect", "slang", "grammar", "abbreviations", "typos"], k=k)
    order = ["indirect", "slang", "grammar", "abbreviations", "typos"]
    out = text
    for nt in order:
        if nt in chosen:
            out = NOISE_FN_MAP[nt](out, severity=severity, rng=rng)
    return out


NOISE_FN_MAP = {
    "typos": inject_typos,
    "abbreviations": inject_abbreviations,
    "slang": inject_slang,
    "grammar": inject_grammatical_errors,
    "indirect": inject_indirect_language,
    "combined": inject_combined_noise,
}
NOISE_TYPES = ["typos", "abbreviations", "slang", "grammar", "indirect", "combined"]


def create_noisy_dataframe(source_df: pd.DataFrame, noise_type: str, severity: str = "medium", seed: int = 42):
    rng = random.Random(seed)
    fn = NOISE_FN_MAP[noise_type]
    noisy_texts = [fn(t, severity=severity, rng=rng) for t in source_df["text"].tolist()]
    out = source_df.copy()
    out["text"] = noisy_texts
    out["noise_type"] = noise_type
    out["severity"] = severity
    return out


# ── Showcase: All 5 Noise Types + Combined on Sample Utterances ──
sample_rows = test_df.sample(3, random_state=CONFIG["seed"])
showcase_records = []
for idx, row in sample_rows.iterrows():
    rng_demo = random.Random(100 + idx)
    rec = {"Intent": row["intent"], "Clean Original": row["text"]}
    for nt in NOISE_TYPES:
        rec[nt.capitalize()] = NOISE_FN_MAP[nt](row["text"], severity="medium", rng=random.Random(100 + idx))
    showcase_records.append(rec)

showcase_df = pd.DataFrame(showcase_records)
pd.set_option("display.max_colwidth", 140)
print("Sample Transformations Across All 5 Noise Types + Combined (Medium Severity):")
display(showcase_df.T)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Build Standardized Evaluation Benchmarks + Augmented Training Sets
# ═══════════════════════════════════════════════════════════════════════════════
# 1. Standardized Test Benchmarks (Medium severity — primary evaluation suite)
TEST_SUITES = {"clean": test_df.copy()}
for i, nt in enumerate(NOISE_TYPES):
    TEST_SUITES[nt] = create_noisy_dataframe(test_df, nt, severity="medium", seed=CONFIG["seed"] + 100 + i)

# 2. Severity Sweep Test Benchmarks (Low / Medium / High on Combined Noise)
SEVERITY_SUITES = {
    sev: create_noisy_dataframe(test_df, "combined", severity=sev, seed=CONFIG["seed"] + 500 + j)
    for j, sev in enumerate(["low", "medium", "high"])
}

# 3. Phase 2 Full 5-Noise Mixed-Severity Augmented Training Set (STRICTLY from train_df!)
def build_augmented_train_set(clean_train_df: pd.DataFrame, allowed_noises: list, copies_per_clean: int = 2, seed: int = 42):
    rng = random.Random(seed)
    parts = [clean_train_df[["text", "label", "intent"]].assign(noise_source="clean")]
    severities = ["low", "medium", "high"]
    sev_weights = [0.25, 0.50, 0.25]

    for c in range(copies_per_clean):
        aug_texts = []
        aug_sources = []
        for t in clean_train_df["text"].tolist():
            nt = rng.choice(allowed_noises)
            sev = rng.choices(severities, weights=sev_weights, k=1)[0]
            aug_texts.append(NOISE_FN_MAP[nt](t, severity=sev, rng=rng))
            aug_sources.append(f"{nt}_{sev}")
        aug_df = clean_train_df[["label", "intent"]].copy()
        aug_df["text"] = aug_texts
        aug_df["noise_source"] = aug_sources
        parts.append(aug_df[["text", "label", "intent", "noise_source"]])

    combined = pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return combined

# Phase 1 Partial Augmentation (Typos + Abbreviations only — for Ablation Study)
train_aug_phase1_df = build_augmented_train_set(
    train_df, allowed_noises=["typos", "abbreviations"], copies_per_clean=1, seed=CONFIG["seed"] + 1
)

# Phase 2 Full Augmentation (All 5 Noise Types + Combined)
train_aug_phase2_df = build_augmented_train_set(
    train_df, allowed_noises=NOISE_TYPES, copies_per_clean=CONFIG["aug_copies_per_clean"], seed=CONFIG["seed"] + 2
)

# Augmented Validation Set (50% clean + 50% mixed noisy) for checkpoint selection of robust models
val_noisy_mix = build_augmented_train_set(val_df, allowed_noises=NOISE_TYPES, copies_per_clean=1, seed=CONFIG["seed"] + 3)

print(f"[DATA] Clean Train Set:            {len(train_df):,} rows")
print(f"[DATA] Phase 1 Aug Train (2-type): {len(train_aug_phase1_df):,} rows")
print(f"[DATA] Phase 2 Aug Train (5-type): {len(train_aug_phase2_df):,} rows")
print(f"[DATA] Mixed Robust Val Set:       {len(val_noisy_mix):,} rows")
print(f"[DATA] Test Suites Prepared:       {list(TEST_SUITES.keys())} ({len(test_df):,} each)")

## 5. Evaluation Metrics, Confidence Calibration (ECE) & Experiment Tracker

To ensure a fair, apples-to-apples comparison across all 3 modeling paradigms (Traditional ML, BiLSTM, DistilBERT), every model is evaluated using the exact same helper functions computing:
- **Accuracy**, **Macro-F1**, **Weighted-F1**, **Macro-Precision**, **Macro-Recall**
- **Expected Calibration Error (ECE)** — measures whether a model's predicted confidence matches its true accuracy
- **Inference Latency (ms / query)** — measures real-time production readiness

In [ ]:
def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 10) -> float:
    """Compute Expected Calibration Error (ECE)."""
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == labels).astype(float)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    n = len(labels)
    for b in range(n_bins):
        lo, hi = bin_edges[b], bin_edges[b + 1]
        mask = (confidences > lo) & (confidences <= hi) if b > 0 else (confidences >= lo) & (confidences <= hi)
        if mask.any():
            ece += (mask.sum() / n) * abs(accuracies[mask].mean() - confidences[mask].mean())
    return float(ece)


def evaluate_predictions(y_true: np.ndarray, probs: np.ndarray):
    y_pred = probs.argmax(axis=1)
    acc = accuracy_score(y_true, y_pred)
    p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    _, _, f1_wt, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    ece = compute_ece(probs, y_true)
    return {
        "accuracy": float(acc),
        "macro_f1": float(f1_mac),
        "weighted_f1": float(f1_wt),
        "macro_precision": float(p_mac),
        "macro_recall": float(r_mac),
        "ece": float(ece),
        "y_pred": y_pred,
        "confidences": probs.max(axis=1),
        "probs": probs,
    }


# Master storage for all models, benchmark results, and experiment log
ALL_RESULTS = defaultdict(dict)   # ALL_RESULTS[model_name][suite_name] = metrics_dict
EXPERIMENT_LOG = []               # M3/M4 Experiment Log rows
MODEL_PREDICTORS = {}             # model_name -> callable(list[str]) -> probs (N, C)

## 6. M3 — Traditional ML Baselines (TF-IDF + Logistic Regression & Calibrated Linear SVM)

We build strong traditional ML baselines using a **hybrid Word (1–2 gram) + Character (3–5 gram) TF-IDF** vectorizer. Including character n-grams gives traditional ML a fair fighting chance against typos and abbreviations. We train both:
1. **TF-IDF + Logistic Regression (Clean & Noise-Augmented)**
2. **TF-IDF + Calibrated Linear SVM (Clean & Noise-Augmented)**

In [ ]:
def make_tfidf_union():
    return FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True, lowercase=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=35000, sublinear_tf=True, lowercase=True)),
    ])


def train_and_eval_sklearn(model_name: str, clf, tr_df: pd.DataFrame, notes: str):
    t0 = time.time()
    vec = make_tfidf_union()
    X_tr = vec.fit_transform(tr_df["text"].tolist())
    y_tr = tr_df["label"].values
    clf.fit(X_tr, y_tr)
    train_sec = time.time() - t0

    def predict_proba_fn(texts):
        X = vec.transform(texts)
        return clf.predict_proba(X)

    MODEL_PREDICTORS[model_name] = predict_proba_fn

    # Measure inference latency on 200 queries
    sample_q = test_df["text"].tolist()[:200]
    t_inf0 = time.time()
    _ = predict_proba_fn(sample_q)
    latency_ms = (time.time() - t_inf0) / len(sample_q) * 1000.0

    # Evaluate across all 7 test suites
    for suite_name, s_df in TEST_SUITES.items():
        probs = predict_proba_fn(s_df["text"].tolist())
        ALL_RESULTS[model_name][suite_name] = evaluate_predictions(s_df["label"].values, probs)

    clean_acc = ALL_RESULTS[model_name]["clean"]["accuracy"] * 100
    comb_acc = ALL_RESULTS[model_name]["combined"]["accuracy"] * 100
    avg_noisy = np.mean([ALL_RESULTS[model_name][nt]["accuracy"] * 100 for nt in NOISE_TYPES[:5]])

    EXPERIMENT_LOG.append({
        "Model": model_name,
        "Paradigm": "Traditional ML",
        "Train Rows": len(tr_df),
        "Train Time (s)": round(train_sec, 1),
        "Latency (ms/q)": round(latency_ms, 3),
        "Clean Acc (%)": round(clean_acc, 2),
        "Avg 5-Noise Acc (%)": round(avg_noisy, 2),
        "Combined Noise Acc (%)": round(comb_acc, 2),
        "Notes": notes,
    })
    print(f"[{model_name}] Train: {train_sec:.1f}s | Clean: {clean_acc:.2f}% | "
          f"Avg 5-Noise: {avg_noisy:.2f}% | Combined: {comb_acc:.2f}% | Latency: {latency_ms:.3f} ms/q")
    return vec, clf


print("Training Traditional ML Models...")
vec_lr_clean, clf_lr_clean = train_and_eval_sklearn(
    "LogReg (Clean)",
    LogisticRegression(max_iter=1000, C=4.0, n_jobs=-1, random_state=CONFIG["seed"]),
    train_df,
    "Word(1-2)+Char(3-5) TF-IDF, Clean Train",
)
vec_lr_aug, clf_lr_aug = train_and_eval_sklearn(
    "LogReg (Robust)",
    LogisticRegression(max_iter=1000, C=4.0, n_jobs=-1, random_state=CONFIG["seed"]),
    train_aug_phase2_df,
    "Word(1-2)+Char(3-5) TF-IDF, 5-Noise Aug",
)
vec_svm_clean, clf_svm_clean = train_and_eval_sklearn(
    "LinearSVM (Clean)",
    CalibratedClassifierCV(LinearSVC(C=1.0, random_state=CONFIG["seed"]), cv=3),
    train_df,
    "Calibrated LinearSVC, Clean Train",
)
vec_svm_aug, clf_svm_aug = train_and_eval_sklearn(
    "LinearSVM (Robust)",
    CalibratedClassifierCV(LinearSVC(C=1.0, random_state=CONFIG["seed"]), cv=3),
    train_aug_phase2_df,
    "Calibrated LinearSVC, 5-Noise Aug",
)

## 7. M3 — Deep Learning Sequence Model (BiLSTM + Attention)

Next, we train a **2-layer Bidirectional LSTM with Additive Self-Attention** in PyTorch. Unlike bag-of-n-grams TF-IDF, BiLSTM models sequential word order and long-range context (helpful for indirect language). We train both **BiLSTM (Clean)** and **BiLSTM (Robust — 5-Noise Augmented)** and record training curves.

In [ ]:
class SimpleVocab:
    def __init__(self, max_size: int = 15000, min_freq: int = 1):
        self.max_size = max_size
        self.min_freq = min_freq
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}

    @staticmethod
    def tokenize(text: str):
        return re.findall(r"\{\{.*?\}\}|[a-z0-9']+|[?!]", text.lower())

    def fit(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(self.tokenize(t))
        for word, freq in counter.most_common(self.max_size - 2):
            if freq >= self.min_freq:
                self.word2idx[word] = len(self.word2idx)
        return self

    def encode(self, text: str, max_len: int = 64):
        toks = self.tokenize(text)[:max_len]
        ids = [self.word2idx.get(tok, 1) for tok in toks]
        length = max(len(ids), 1)
        if len(ids) < max_len:
            ids += [0] * (max_len - len(ids))
        return ids, length

    def __len__(self):
        return len(self.word2idx)


class SeqDataset(Dataset):
    def __init__(self, texts, labels, vocab: SimpleVocab, max_len: int = 64):
        self.encoded = [vocab.encode(t, max_len) for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        ids, length = self.encoded[idx]
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(length, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )


class BiLSTMAttentionClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, num_layers: int, num_classes: int, dropout: float = 0.35):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embed_drop = nn.Dropout(dropout * 0.5)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0.0
        )
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x, lengths=None):
        mask = (x != 0)  # (B, L)
        emb = self.embed_drop(self.embedding(x))
        out, _ = self.lstm(emb)  # (B, L, 2H)
        attn_scores = self.attn(out).squeeze(-1)  # (B, L)
        attn_scores = attn_scores.masked_fill(~mask, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=-1).unsqueeze(-1)  # (B, L, 1)
        context = (out * attn_weights).sum(dim=1)  # (B, 2H)
        return self.classifier(context)


def train_and_eval_bilstm(model_name: str, tr_df: pd.DataFrame, vl_df: pd.DataFrame, notes: str):
    set_seed(CONFIG["seed"])
    t0 = time.time()
    slug = re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    ckpt_path = OUT_DIR / "checkpoints" / f"{slug}_ckpt.pt"
    max_len = 64
    total_epochs = CONFIG["bilstm_epochs"]

    # Load existing checkpoint if available and resume is enabled
    ckpt = None
    if CONFIG["resume_from_checkpoints"] and ckpt_path.exists():
        try:
            ckpt = load_checkpoint_safe(ckpt_path)
            print(f"  [RESUME] Found checkpoint for {model_name}: {ckpt_path} "
                  f"(epoch {ckpt['epoch']}/{ckpt['total_epochs']}, completed={ckpt.get('completed', False)})")
        except Exception as e:
            print(f"  [WARN] Could not load checkpoint {ckpt_path} ({e}); starting fresh.")
            ckpt = None

    vocab = SimpleVocab(max_size=CONFIG["bilstm_vocab_size"])
    if ckpt is not None and "vocab_word2idx" in ckpt:
        vocab.word2idx = ckpt["vocab_word2idx"]
    else:
        vocab.fit(tr_df["text"].tolist())

    model = BiLSTMAttentionClassifier(
        vocab_size=len(vocab),
        embed_dim=CONFIG["bilstm_embed_dim"],
        hidden_dim=CONFIG["bilstm_hidden_dim"],
        num_layers=CONFIG["bilstm_num_layers"],
        num_classes=NUM_CLASSES,
        dropout=CONFIG["bilstm_dropout"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["bilstm_lr"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])

    best_val_loss = float("inf")
    best_state = None
    history = []
    start_epoch = 1
    prior_elapsed = 0.0

    if ckpt is not None:
        model.load_state_dict(ckpt["model_state_dict"])
        best_val_loss = ckpt["best_val_loss"]
        best_state = ckpt["best_state"]
        history = ckpt.get("history", [])
        prior_elapsed = ckpt.get("elapsed_sec", 0.0)
        if ckpt.get("completed", False) or ckpt["epoch"] >= total_epochs:
            start_epoch = total_epochs + 1
            print(f"  [SKIP] {model_name} already completed training — loaded best weights directly.")
        else:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            start_epoch = ckpt["epoch"] + 1
            print(f"  [RESUME] Continuing {model_name} from epoch {start_epoch}/{total_epochs}...")

    if start_epoch <= total_epochs:
        tr_ds = SeqDataset(tr_df["text"].tolist(), tr_df["label"].values, vocab, max_len)
        vl_ds = SeqDataset(vl_df["text"].tolist(), vl_df["label"].values, vocab, max_len)
        tr_loader = DataLoader(tr_ds, batch_size=CONFIG["bilstm_batch_size"], shuffle=True)
        vl_loader = DataLoader(vl_ds, batch_size=128, shuffle=False)

        for epoch in range(start_epoch, total_epochs + 1):
            model.train()
            tr_loss_sum, tr_n = 0.0, 0
            for bx, blen, by in tr_loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                optimizer.zero_grad()
                logits = model(bx, blen)
                loss = criterion(logits, by)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                tr_loss_sum += loss.item() * bx.size(0)
                tr_n += bx.size(0)

            model.eval()
            vl_loss_sum, vl_correct, vl_n = 0.0, 0, 0
            with torch.no_grad():
                for bx, blen, by in vl_loader:
                    bx, by = bx.to(DEVICE), by.to(DEVICE)
                    logits = model(bx, blen)
                    loss = criterion(logits, by)
                    vl_loss_sum += loss.item() * bx.size(0)
                    vl_correct += (logits.argmax(dim=-1) == by).sum().item()
                    vl_n += bx.size(0)

            tr_loss = tr_loss_sum / tr_n
            vl_loss = vl_loss_sum / vl_n
            vl_acc = vl_correct / vl_n
            scheduler.step(vl_loss)
            history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": vl_loss, "val_acc": vl_acc})

            if vl_loss < best_val_loss:
                best_val_loss = vl_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

            # Save atomic checkpoint after every epoch
            save_checkpoint_atomic({
                "epoch": epoch,
                "total_epochs": total_epochs,
                "completed": (epoch == total_epochs),
                "model_state_dict": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_loss": best_val_loss,
                "best_state": best_state,
                "history": history,
                "vocab_word2idx": vocab.word2idx,
                "elapsed_sec": prior_elapsed + (time.time() - t0),
            }, ckpt_path)
            print(f"    [{model_name}] Ep {epoch:2d}/{total_epochs} | Val Loss: {vl_loss:.4f} | "
                  f"Val Acc: {vl_acc*100:.2f}% | Checkpoint saved")

    model.load_state_dict(best_state)
    model.to(DEVICE).eval()
    train_sec = prior_elapsed + (time.time() - t0)

    def predict_proba_fn(texts):
        model.eval()
        ds = SeqDataset(texts, [0] * len(texts), vocab, max_len)
        loader = DataLoader(ds, batch_size=256, shuffle=False)
        probs_list = []
        with torch.no_grad():
            for bx, blen, _ in loader:
                logits = model(bx.to(DEVICE), blen)
                probs_list.append(torch.softmax(logits, dim=-1).cpu().numpy())
        return np.vstack(probs_list)

    MODEL_PREDICTORS[model_name] = predict_proba_fn

    # Measure inference latency
    sample_q = test_df["text"].tolist()[:200]
    t_inf0 = time.time()
    _ = predict_proba_fn(sample_q)
    latency_ms = (time.time() - t_inf0) / len(sample_q) * 1000.0

    for suite_name, s_df in TEST_SUITES.items():
        probs = predict_proba_fn(s_df["text"].tolist())
        ALL_RESULTS[model_name][suite_name] = evaluate_predictions(s_df["label"].values, probs)

    clean_acc = ALL_RESULTS[model_name]["clean"]["accuracy"] * 100
    comb_acc = ALL_RESULTS[model_name]["combined"]["accuracy"] * 100
    avg_noisy = np.mean([ALL_RESULTS[model_name][nt]["accuracy"] * 100 for nt in NOISE_TYPES[:5]])

    EXPERIMENT_LOG.append({
        "Model": model_name,
        "Paradigm": "Deep Learning (RNN)",
        "Train Rows": len(tr_df),
        "Train Time (s)": round(train_sec, 1),
        "Latency (ms/q)": round(latency_ms, 3),
        "Clean Acc (%)": round(clean_acc, 2),
        "Avg 5-Noise Acc (%)": round(avg_noisy, 2),
        "Combined Noise Acc (%)": round(comb_acc, 2),
        "Notes": notes,
    })
    print(f"[{model_name}] Train: {train_sec:.1f}s | Clean: {clean_acc:.2f}% | "
          f"Avg 5-Noise: {avg_noisy:.2f}% | Combined: {comb_acc:.2f}% | Latency: {latency_ms:.3f} ms/q")
    return model, vocab, history


print("Training BiLSTM + Attention Models...")
bilstm_clean, vocab_clean, hist_bilstm_clean = train_and_eval_bilstm(
    "BiLSTM (Clean)", train_df, val_df, "2-Layer BiLSTM+Attn, Clean Train"
)
bilstm_robust, vocab_robust, hist_bilstm_robust = train_and_eval_bilstm(
    "BiLSTM (Robust)", train_aug_phase2_df, val_noisy_mix, "2-Layer BiLSTM+Attn, 5-Noise Aug"
)

## 8. M3 & M4 — Fine-Tuned DistilBERT Models & Phase 1 vs Phase 2 Ablation

We fine-tune `distilbert-base-uncased` (~66M parameters, 40% smaller and 60% faster than BERT-base while retaining 97% of its language understanding) across **three controlled experimental configurations** to directly validate our Phase 2 hypothesis:

1. **DistilBERT (Clean — Phase 1 Baseline):** Trained strictly on clean Bitext utterances.
2. **DistilBERT (Phase 1 Partial-Aug — Ablation):** Trained with only Phase 1 noise types (*Typos + Abbreviations*).
3. **DistilBERT (Phase 2 Robust — Final Model):** Trained with **all 5 noise types + combined multi-noise** at mixed severities (`low/medium/high`), **label smoothing (0.05)**, and **post-hoc Temperature Scaling** on the validation set for calibrated production confidence scores!

In [ ]:
TOKENIZER = AutoTokenizer.from_pretrained(CONFIG["transformer_model"])


class TransformerDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len: int = 96):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodings["input_ids"][idx],
            "attention_mask": self.encodings["attention_mask"][idx],
            "labels": self.labels[idx],
        }


def get_transformer_logits(model, texts, batch_size: int = 128):
    model.eval()
    all_logits = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        enc = TOKENIZER(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=CONFIG["max_seq_length"],
            return_tensors="pt",
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            if USE_AMP:
                with torch.cuda.amp.autocast():
                    logits = model(**enc).logits
            else:
                logits = model(**enc).logits
        all_logits.append(logits.float().cpu().numpy())
    return np.vstack(all_logits)


def fit_temperature_scaling(val_logits: np.ndarray, val_labels: np.ndarray) -> float:
    """Optimize scalar temperature T > 0 to minimize validation NLL (improves confidence calibration)."""
    logits_t = torch.tensor(val_logits, dtype=torch.float32)
    labels_t = torch.tensor(val_labels, dtype=torch.long)
    log_temp = nn.Parameter(torch.zeros(1))
    optimizer = torch.optim.LBFGS([log_temp], lr=0.1, max_iter=50)

    def closure():
        optimizer.zero_grad()
        temp = torch.exp(log_temp).clamp(0.1, 10.0)
        loss = F.cross_entropy(logits_t / temp, labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    best_temp = float(torch.exp(log_temp).clamp(0.1, 10.0).item())
    return best_temp


def train_and_eval_distilbert(
    model_name: str,
    tr_df: pd.DataFrame,
    vl_df: pd.DataFrame,
    epochs: int,
    label_smoothing: float,
    calibrate_temp: bool,
    save_subdir: str,
    notes: str,
):
    set_seed(CONFIG["seed"])
    t0 = time.time()
    ckpt_path = OUT_DIR / "checkpoints" / f"{save_subdir}_ckpt.pt"

    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["transformer_model"],
        num_labels=NUM_CLASSES,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CONFIG["bert_lr"], weight_decay=CONFIG["bert_weight_decay"]
    )
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    scaler = torch.cuda.amp.GradScaler() if USE_AMP else None

    best_val_loss = float("inf")
    best_state = None
    history = []
    start_epoch = 1
    prior_elapsed = 0.0
    saved_temp = None
    saved_ece_before = None
    saved_ece_after = None

    # Check for existing per-epoch checkpoint to resume after a crash
    ckpt = None
    if CONFIG["resume_from_checkpoints"] and ckpt_path.exists():
        try:
            ckpt = load_checkpoint_safe(ckpt_path)
            print(f"\n[RESUME] Found checkpoint for {model_name}: {ckpt_path} "
                  f"(epoch {ckpt['epoch']}/{ckpt['total_epochs']}, completed={ckpt.get('completed', False)})")
        except Exception as e:
            print(f"\n[WARN] Failed to load checkpoint {ckpt_path} ({e}); starting fresh.")
            ckpt = None

    if ckpt is not None:
        model.load_state_dict(ckpt["model_state_dict"])
        best_val_loss = ckpt["best_val_loss"]
        best_state = ckpt["best_state"]
        history = ckpt.get("history", [])
        prior_elapsed = ckpt.get("elapsed_sec", 0.0)
        saved_temp = ckpt.get("temperature")
        saved_ece_before = ckpt.get("ece_before_cal")
        saved_ece_after = ckpt.get("ece_after_cal")
        if ckpt.get("completed", False) or ckpt["epoch"] >= epochs:
            start_epoch = epochs + 1
            print(f"  [SKIP] {model_name} already finished all {epochs} epochs — loaded best weights directly!")
        else:
            start_epoch = ckpt["epoch"] + 1

    if start_epoch <= epochs:
        tr_ds = TransformerDataset(tr_df["text"].tolist(), tr_df["label"].values, TOKENIZER, CONFIG["max_seq_length"])
        vl_ds = TransformerDataset(vl_df["text"].tolist(), vl_df["label"].values, TOKENIZER, CONFIG["max_seq_length"])

        tr_loader = DataLoader(tr_ds, batch_size=CONFIG["bert_batch_size"], shuffle=True, pin_memory=(DEVICE.type == "cuda"))
        vl_loader = DataLoader(vl_ds, batch_size=128, shuffle=False, pin_memory=(DEVICE.type == "cuda"))

        total_steps = len(tr_loader) * epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(total_steps * CONFIG["bert_warmup_ratio"]),
            num_training_steps=total_steps,
        )

        # Restore optimizer, scheduler, and AMP scaler if resuming mid-training
        if ckpt is not None and start_epoch > 1:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            if USE_AMP and scaler is not None and ckpt.get("scaler_state_dict") is not None:
                scaler.load_state_dict(ckpt["scaler_state_dict"])
            print(f"  [RESUME] Restored optimizer/scheduler/scaler — continuing from epoch {start_epoch}/{epochs}")

        print(f"\n{'═' * 65}")
        print(f"Training {model_name} ({len(tr_df):,} train rows, epochs {start_epoch}..{epochs})")
        print(f"{'═' * 65}")

        try:
            for epoch in range(start_epoch, epochs + 1):
                ep_t0 = time.time()
                model.train()
                tr_loss_sum, tr_n = 0.0, 0

                for batch in tqdm(tr_loader, desc=f"{model_name} Ep {epoch}/{epochs}", leave=False):
                    input_ids = batch["input_ids"].to(DEVICE)
                    attention_mask = batch["attention_mask"].to(DEVICE)
                    labels = batch["labels"].to(DEVICE)
                    optimizer.zero_grad()

                    if USE_AMP:
                        with torch.cuda.amp.autocast():
                            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                            loss = criterion(logits, labels)
                        scaler.scale(loss).backward()
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                        loss = criterion(logits, labels)
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()

                    scheduler.step()
                    tr_loss_sum += loss.item() * input_ids.size(0)
                    tr_n += input_ids.size(0)

                # Validation
                model.eval()
                vl_loss_sum, vl_correct, vl_n = 0.0, 0, 0
                with torch.no_grad():
                    for batch in vl_loader:
                        input_ids = batch["input_ids"].to(DEVICE)
                        attention_mask = batch["attention_mask"].to(DEVICE)
                        labels = batch["labels"].to(DEVICE)
                        if USE_AMP:
                            with torch.cuda.amp.autocast():
                                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                                loss = criterion(logits, labels)
                        else:
                            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                            loss = criterion(logits, labels)
                        vl_loss_sum += loss.item() * input_ids.size(0)
                        vl_correct += (logits.argmax(dim=-1) == labels).sum().item()
                        vl_n += input_ids.size(0)

                tr_loss = tr_loss_sum / tr_n
                vl_loss = vl_loss_sum / vl_n
                vl_acc = vl_correct / vl_n
                ep_sec = time.time() - ep_t0
                history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": vl_loss, "val_acc": vl_acc})

                is_best = ""
                if vl_loss < best_val_loss:
                    best_val_loss = vl_loss
                    best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                    is_best = " ★"

                # Atomic checkpoint save after EVERY epoch (crash-safe for SageMaker!)
                save_checkpoint_atomic({
                    "epoch": epoch,
                    "total_epochs": epochs,
                    "completed": (epoch == epochs),
                    "model_state_dict": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "scaler_state_dict": scaler.state_dict() if (USE_AMP and scaler is not None) else None,
                    "best_val_loss": best_val_loss,
                    "best_state": best_state,
                    "history": history,
                    "elapsed_sec": prior_elapsed + (time.time() - t0),
                }, ckpt_path)

                print(f"  Epoch {epoch}/{epochs} | Train Loss: {tr_loss:.4f} | Val Loss: {vl_loss:.4f} | "
                      f"Val Acc: {vl_acc*100:.2f}% | {ep_sec:.1f}s | Checkpoint saved{is_best}")
        except KeyboardInterrupt:
            print(f"\n  [INTERRUPT] Stopped manually. Last completed epoch checkpoint is safe at: {ckpt_path}")
            if best_state is None:
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.to(DEVICE).eval()
    train_sec = prior_elapsed + (time.time() - t0)

    # Optional Temperature Calibration on Validation Set (reuses cached value on resume)
    temperature = 1.0
    ece_before_cal, ece_after_cal = None, None
    if calibrate_temp:
        if saved_temp is not None:
            temperature = saved_temp
            ece_before_cal = saved_ece_before
            ece_after_cal = saved_ece_after
            print(f"  [CALIBRATION] Loaded cached Temperature T = {temperature:.3f} | "
                  f"Val ECE: {ece_before_cal:.4f} -> {ece_after_cal:.4f}")
        else:
            val_logits = get_transformer_logits(model, vl_df["text"].tolist())
            raw_probs = torch.softmax(torch.tensor(val_logits), dim=-1).numpy()
            ece_before_cal = compute_ece(raw_probs, vl_df["label"].values)
            temperature = fit_temperature_scaling(val_logits, vl_df["label"].values)
            cal_probs = torch.softmax(torch.tensor(val_logits / temperature), dim=-1).numpy()
            ece_after_cal = compute_ece(cal_probs, vl_df["label"].values)
            print(f"  [CALIBRATION] Optimal Temperature T = {temperature:.3f} | "
                  f"Val ECE: {ece_before_cal:.4f} -> {ece_after_cal:.4f}")
            # Persist temperature in the final checkpoint too
            if ckpt_path.exists():
                final_ckpt = load_checkpoint_safe(ckpt_path)
                final_ckpt["temperature"] = temperature
                final_ckpt["ece_before_cal"] = ece_before_cal
                final_ckpt["ece_after_cal"] = ece_after_cal
                save_checkpoint_atomic(final_ckpt, ckpt_path)

    # Save model & tokenizer
    if save_subdir:
        save_path = OUT_DIR / "models" / save_subdir
        model.save_pretrained(save_path)
        TOKENIZER.save_pretrained(save_path)
        with open(save_path / "calibration.json", "w") as f:
            json.dump({"temperature": temperature, "ece_before": ece_before_cal, "ece_after": ece_after_cal}, f, indent=2)

    def predict_proba_fn(texts, temp=temperature, mdl=model):
        logits = get_transformer_logits(mdl, texts)
        return torch.softmax(torch.tensor(logits / temp), dim=-1).numpy()

    MODEL_PREDICTORS[model_name] = predict_proba_fn

    # Measure inference latency (single-batch realistic throughput)
    sample_q = test_df["text"].tolist()[:200]
    t_inf0 = time.time()
    _ = predict_proba_fn(sample_q)
    latency_ms = (time.time() - t_inf0) / len(sample_q) * 1000.0

    for suite_name, s_df in TEST_SUITES.items():
        probs = predict_proba_fn(s_df["text"].tolist())
        ALL_RESULTS[model_name][suite_name] = evaluate_predictions(s_df["label"].values, probs)

    clean_acc = ALL_RESULTS[model_name]["clean"]["accuracy"] * 100
    comb_acc = ALL_RESULTS[model_name]["combined"]["accuracy"] * 100
    avg_noisy = np.mean([ALL_RESULTS[model_name][nt]["accuracy"] * 100 for nt in NOISE_TYPES[:5]])

    EXPERIMENT_LOG.append({
        "Model": model_name,
        "Paradigm": "Transformer",
        "Train Rows": len(tr_df),
        "Train Time (s)": round(train_sec, 1),
        "Latency (ms/q)": round(latency_ms, 3),
        "Clean Acc (%)": round(clean_acc, 2),
        "Avg 5-Noise Acc (%)": round(avg_noisy, 2),
        "Combined Noise Acc (%)": round(comb_acc, 2),
        "Notes": notes,
    })
    print(f"[{model_name}] DONE | Clean: {clean_acc:.2f}% | Avg 5-Noise: {avg_noisy:.2f}% | "
          f"Combined: {comb_acc:.2f}% | Latency: {latency_ms:.3f} ms/q")
    return model, history, temperature


# 1. Phase 1 Baseline: DistilBERT trained on Clean data only
bert_clean, hist_bert_clean, temp_clean = train_and_eval_distilbert(
    model_name="DistilBERT (Clean)",
    tr_df=train_df,
    vl_df=val_df,
    epochs=CONFIG["bert_epochs"],
    label_smoothing=0.0,
    calibrate_temp=False,
    save_subdir="distilbert_clean",
    notes="Phase 1 Baseline (Clean Train Only)",
)

# 2. Ablation Model: DistilBERT trained on Phase 1 Partial Noise (Typos + Abbrev only)
bert_p1_aug, hist_bert_p1, temp_p1 = train_and_eval_distilbert(
    model_name="DistilBERT (P1-Aug)",
    tr_df=train_aug_phase1_df,
    vl_df=val_df,
    epochs=max(2, CONFIG["bert_epochs"] - 1),
    label_smoothing=0.0,
    calibrate_temp=False,
    save_subdir="distilbert_p1_partial_aug",
    notes="Ablation: Phase 1 Aug (Typos + Abbrev only)",
)

# 3. Phase 2 Final Model: DistilBERT trained on ALL 5 Noise Types + Label Smoothing + Temp Calibration
bert_robust, hist_bert_robust, temp_robust = train_and_eval_distilbert(
    model_name="DistilBERT (Robust)",
    tr_df=train_aug_phase2_df,
    vl_df=val_noisy_mix,
    epochs=CONFIG["bert_epochs"],
    label_smoothing=CONFIG["label_smoothing"],
    calibrate_temp=True,
    save_subdir="distilbert_robust_final",
    notes="Phase 2 Final: 5-Noise Mixed-Sev Aug + Label Smoothing + Temp Cal",
)

## 9. M4 — Comprehensive Evaluation, Robustness Matrix & Ablation Study

Here we consolidate all quantitative results across:
1. **Experiment Log** (Model Paradigm, Training Time, Latency, Clean vs Noisy Accuracy)
2. **Full Robustness Matrix** (All Models × Clean + 5 Noise Types + Combined Multi-Noise)
3. **Training Curves** (BiLSTM vs DistilBERT convergence)
4. **Noise Severity Sensitivity Sweep** (`Low` vs `Medium` vs `High` Combined Noise)
5. **Phase 1 → Phase 2 Ablation Study** (Quantifying the exact gain from expanding from 2 noise types to all 5 noise types + calibration)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9.1 Experiment Log & Full Robustness Comparison Tables
# ═══════════════════════════════════════════════════════════════════════════════
exp_log_df = pd.DataFrame(EXPERIMENT_LOG)
exp_log_df.to_csv(OUT_DIR / "tables" / "experiment_log.csv", index=False)

print("═" * 80)
print("M3/M4 EXPERIMENT LOG")
print("═" * 80)
display(exp_log_df)

# Full Accuracy Matrix (Models x Test Suites)
suite_order = ["clean", "typos", "abbreviations", "slang", "grammar", "indirect", "combined"]
acc_matrix = pd.DataFrame(
    {suite: {m: ALL_RESULTS[m][suite]["accuracy"] * 100 for m in ALL_RESULTS} for suite in suite_order}
)
acc_matrix["Avg 5-Noise"] = acc_matrix[NOISE_TYPES[:5]].mean(axis=1)
acc_matrix["Retention (Comb/Clean)"] = (acc_matrix["combined"] / acc_matrix["clean"]) * 100
acc_matrix = acc_matrix.round(2)
acc_matrix.to_csv(OUT_DIR / "tables" / "robustness_accuracy_matrix.csv")

f1_matrix = pd.DataFrame(
    {suite: {m: ALL_RESULTS[m][suite]["macro_f1"] * 100 for m in ALL_RESULTS} for suite in suite_order}
).round(2)
f1_matrix.to_csv(OUT_DIR / "tables" / "robustness_macro_f1_matrix.csv")

print("\n" + "═" * 80)
print("ROBUSTNESS ACCURACY MATRIX (%) — ALL MODELS × ALL NOISE TYPES")
print("═" * 80)
display(acc_matrix)

print("\n" + "═" * 80)
print("ROBUSTNESS MACRO-F1 MATRIX (%)")
print("═" * 80)
display(f1_matrix)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9.2 Visual Robustness Benchmarks (Heatmap, Grouped Bars, Degradation & Curves)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(20, 6))

# Left: Heatmap of Accuracy across Models x Noise Types
sns.heatmap(
    acc_matrix[suite_order],
    annot=True, fmt=".1f", cmap="YlGnBu", vmin=50, vmax=100,
    linewidths=0.5, ax=axes[0], cbar_kws={"label": "Accuracy (%)"}
)
axes[0].set_title("Model Accuracy (%) Across Clean & All 5 Noise Types", fontweight="bold", fontsize=13)
axes[0].set_xlabel("Evaluation Suite")
axes[0].set_ylabel("Model")

# Right: Clean vs Robust Improvement (pp) for DistilBERT Phase 1 -> Phase 2
p1_scores = acc_matrix.loc["DistilBERT (Clean)", suite_order]
p1_aug_scores = acc_matrix.loc["DistilBERT (P1-Aug)", suite_order]
p2_scores = acc_matrix.loc["DistilBERT (Robust)", suite_order]

x = np.arange(len(suite_order))
w = 0.26
b1 = axes[1].bar(x - w, p1_scores, width=w, label="DistilBERT (Clean - P1)", color="#e74c3c")
b2 = axes[1].bar(x, p1_aug_scores, width=w, label="DistilBERT (P1 2-Noise Aug)", color="#f39c12")
b3 = axes[1].bar(x + w, p2_scores, width=w, label="DistilBERT (P2 5-Noise Robust)", color="#27ae60")
axes[1].set_xticks(x)
axes[1].set_xticklabels([s.capitalize() for s in suite_order], rotation=25)
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_ylim(max(0, min(p1_scores.min(), p2_scores.min()) - 10), 103)
axes[1].set_title("Ablation Study: Clean vs Phase 1 Aug vs Phase 2 Full 5-Noise Robust", fontweight="bold", fontsize=13)
axes[1].legend(loc="lower left")
for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2, h + 0.5, f"{h:.1f}", ha="center", va="bottom", fontsize=7.5)

plt.tight_layout()
plt.savefig(OUT_DIR / "plots" / "02_robustness_heatmap_and_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 9.3 Paradigm Comparison & Training Curves
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

# 1. Best of each paradigm on Clean vs Avg 5-Noise vs Combined
key_models = ["LogReg (Robust)", "LinearSVM (Robust)", "BiLSTM (Robust)", "DistilBERT (Clean)", "DistilBERT (Robust)"]
plot_df = acc_matrix.loc[key_models, ["clean", "Avg 5-Noise", "combined"]].reset_index().melt(
    id_vars="index", var_name="Benchmark", value_name="Accuracy (%)"
)
sns.barplot(data=plot_df, x="index", y="Accuracy (%)", hue="Benchmark", ax=axes[0], palette="Set2")
axes[0].set_title("Paradigm Comparison: Traditional ML vs BiLSTM vs DistilBERT", fontweight="bold")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=20)
axes[0].set_ylim(50, 103)
axes[0].legend(title="Suite", loc="lower right")

# 2. Training Loss Curves
axes[1].plot([h["epoch"] for h in hist_bilstm_robust], [h["train_loss"] for h in hist_bilstm_robust], "o--", label="BiLSTM (Robust) Train")
axes[1].plot([h["epoch"] for h in hist_bilstm_robust], [h["val_loss"] for h in hist_bilstm_robust], "o-", label="BiLSTM (Robust) Val")
axes[1].plot([h["epoch"] for h in hist_bert_robust], [h["train_loss"] for h in hist_bert_robust], "s--", label="DistilBERT (Robust) Train")
axes[1].plot([h["epoch"] for h in hist_bert_robust], [h["val_loss"] for h in hist_bert_robust], "s-", label="DistilBERT (Robust) Val")
axes[1].set_title("Training & Validation Loss Curves", fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Cross-Entropy Loss")
axes[1].legend(fontsize=8)

# 3. Noise Severity Sweep (Low / Medium / High Combined Noise)
sev_records = []
for m_name in ["LogReg (Robust)", "BiLSTM (Robust)", "DistilBERT (Clean)", "DistilBERT (Robust)"]:
    pred_fn = MODEL_PREDICTORS[m_name]
    for sev, s_df in SEVERITY_SUITES.items():
        probs = pred_fn(s_df["text"].tolist())
        acc = accuracy_score(s_df["label"].values, probs.argmax(axis=1)) * 100
        sev_records.append({"Model": m_name, "Severity": sev.capitalize(), "Accuracy (%)": acc})

sev_df = pd.DataFrame(sev_records)
sev_df.to_csv(OUT_DIR / "tables" / "severity_sweep.csv", index=False)
sns.lineplot(data=sev_df, x="Severity", y="Accuracy (%)", hue="Model", marker="o", linewidth=2.5, ax=axes[2])
axes[2].set_title("Stress Test: Combined Noise Severity Sweep", fontweight="bold")
axes[2].set_ylim(max(0, sev_df["Accuracy (%)"].min() - 10), 102)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / "plots" / "03_paradigm_curves_severity.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Confusion Matrices, Confidence Calibration & Failure Analysis ("Where Does It Fail?")

A key requirement for both academic evaluation and production deployment is understanding **exact failure modes**:
1. **27×27 Confusion Matrices** on Clean Test vs Combined Multi-Noise Test for our best model (`DistilBERT (Robust)`).
2. **Per-Class Classification Report** (identifying the hardest vs easiest intents).
3. **Top Confused Intent Pairs** (e.g., sibling intents within the same category such as `cancel_order` vs `change_order`).
4. **Confidence Calibration & Escalation Analysis** — showing how confidence thresholding (`τ`) catches errors and routes uncertain queries to a human agent.
5. **Categorized Failure Analysis** with concrete failure examples, root-cause diagnosis, and **Limitations & Threats to Validity**.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 10.1 Confusion Matrices (Clean vs Combined Noisy) & Per-Class Report
# ═══════════════════════════════════════════════════════════════════════════════
BEST_MODEL = "DistilBERT (Robust)"
clean_res = ALL_RESULTS[BEST_MODEL]["clean"]
comb_res = ALL_RESULTS[BEST_MODEL]["combined"]

cm_clean = confusion_matrix(TEST_SUITES["clean"]["label"].values, clean_res["y_pred"], labels=range(NUM_CLASSES))
cm_comb = confusion_matrix(TEST_SUITES["combined"]["label"].values, comb_res["y_pred"], labels=range(NUM_CLASSES))

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
sns.heatmap(cm_clean, xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, cmap="Blues",
            annot=False, ax=axes[0], cbar_kws={"shrink": 0.7})
axes[0].set_title(f"{BEST_MODEL} — Clean Test Confusion Matrix (Acc: {clean_res['accuracy']*100:.2f}%)", fontweight="bold")
axes[0].set_xlabel("Predicted Intent")
axes[0].set_ylabel("True Intent")
axes[0].tick_params(axis="x", rotation=90, labelsize=7.5)
axes[0].tick_params(axis="y", rotation=0, labelsize=7.5)

sns.heatmap(cm_comb, xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, cmap="OrRd",
            annot=False, ax=axes[1], cbar_kws={"shrink": 0.7})
axes[1].set_title(f"{BEST_MODEL} — Combined Multi-Noise Confusion Matrix (Acc: {comb_res['accuracy']*100:.2f}%)", fontweight="bold")
axes[1].set_xlabel("Predicted Intent")
axes[1].set_ylabel("True Intent")
axes[1].tick_params(axis="x", rotation=90, labelsize=7.5)
axes[1].tick_params(axis="y", rotation=0, labelsize=7.5)

plt.tight_layout()
plt.savefig(OUT_DIR / "plots" / "04_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

# Per-class F1 report on Clean vs Combined Noise
rep_clean = classification_report(
    TEST_SUITES["clean"]["label"].values, clean_res["y_pred"],
    target_names=LABEL_NAMES, output_dict=True, zero_division=0
)
rep_comb = classification_report(
    TEST_SUITES["combined"]["label"].values, comb_res["y_pred"],
    target_names=LABEL_NAMES, output_dict=True, zero_division=0
)
per_class_df = pd.DataFrame([
    {
        "Intent": lbl,
        "Category": INTENT_TO_CATEGORY[lbl],
        "Support": int(rep_clean[lbl]["support"]),
        "Clean F1 (%)": round(rep_clean[lbl]["f1-score"] * 100, 2),
        "Combined Noise F1 (%)": round(rep_comb[lbl]["f1-score"] * 100, 2),
        "Drop (pp)": round((rep_comb[lbl]["f1-score"] - rep_clean[lbl]["f1-score"]) * 100, 2),
    }
    for lbl in LABEL_NAMES
]).sort_values("Combined Noise F1 (%)", ascending=True).reset_index(drop=True)

per_class_df.to_csv(OUT_DIR / "tables" / "per_class_f1_report.csv", index=False)
print("Hardest Intents Under Combined Noise (Bottom 10 by F1):")
display(per_class_df.head(10))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 10.2 Confidence Calibration, Reliability Diagram & Human-Escalation Tradeoff
# ═══════════════════════════════════════════════════════════════════════════════
probs_comb = comb_res["probs"]
y_true_comb = TEST_SUITES["combined"]["label"].values
y_pred_comb = comb_res["y_pred"]
confs_comb = comb_res["confidences"]
is_correct_comb = (y_pred_comb == y_true_comb)

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

# 1. Confidence Distribution: Correct vs Incorrect Predictions
sns.histplot(confs_comb[is_correct_comb], bins=20, color="#27ae60", alpha=0.6, label="Correct", stat="density", ax=axes[0])
if (~is_correct_comb).any():
    sns.histplot(confs_comb[~is_correct_comb], bins=20, color="#e74c3c", alpha=0.6, label="Incorrect", stat="density", ax=axes[0])
axes[0].axvline(CONFIG["default_confidence_threshold"], color="black", linestyle="--",
                label=f"Threshold τ={CONFIG['default_confidence_threshold']}")
axes[0].set_title(f"Confidence Distribution (Combined Noise, ECE={comb_res['ece']:.4f})", fontweight="bold")
axes[0].set_xlabel("Predicted Max Probability")
axes[0].legend()

# 2. Reliability Diagram (Calibration)
bins = np.linspace(0, 1, 11)
bin_accs, bin_confs = [], []
for b in range(10):
    mask = (confs_comb > bins[b]) & (confs_comb <= bins[b + 1]) if b > 0 else (confs_comb >= bins[b]) & (confs_comb <= bins[b + 1])
    if mask.any():
        bin_accs.append(is_correct_comb[mask].mean())
        bin_confs.append(confs_comb[mask].mean())
axes[1].plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
axes[1].plot(bin_confs, bin_accs, "o-", color="#2980b9", linewidth=2, label=f"{BEST_MODEL}")
axes[1].set_title("Reliability Diagram (Combined Noise)", fontweight="bold")
axes[1].set_xlabel("Mean Predicted Confidence")
axes[1].set_ylabel("Empirical Accuracy")
axes[1].legend()

# 3. Selective Prediction: Accuracy vs Automation Coverage across Thresholds
thresholds = np.linspace(0.30, 0.95, 14)
covs, auto_accs = [], []
for tau in thresholds:
    keep = confs_comb >= tau
    covs.append(keep.mean() * 100)
    auto_accs.append(is_correct_comb[keep].mean() * 100 if keep.any() else 100.0)

ax3 = axes[2]
ax3.plot(thresholds, auto_accs, "o-", color="#27ae60", linewidth=2, label="Auto-Routed Accuracy (%)")
ax3.set_xlabel("Confidence Threshold (τ)")
ax3.set_ylabel("Auto-Routed Accuracy (%)", color="#27ae60")
ax3.set_title("Human-Escalation Tradeoff (Combined Noise)", fontweight="bold")
ax3_r = ax3.twinx()
ax3_r.plot(thresholds, covs, "s--", color="#8e44ad", linewidth=2, label="Automation Coverage (%)")
ax3_r.set_ylabel("Automation Coverage (%)", color="#8e44ad")
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / "plots" / "05_calibration_and_escalation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# 10.3 Automated Failure Analysis ("Where Does It Fail?")
# ═══════════════════════════════════════════════════════════════════════════════
# 1. Top Confused Intent Pairs on Combined Multi-Noise
confused_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm_comb[i, j] > 0:
            same_cat = (INTENT_TO_CATEGORY[LABEL_NAMES[i]] == INTENT_TO_CATEGORY[LABEL_NAMES[j]])
            confused_pairs.append({
                "True Intent": LABEL_NAMES[i],
                "Predicted Intent": LABEL_NAMES[j],
                "Error Count": int(cm_comb[i, j]),
                "Same Category (Sibling)?": "Yes" if same_cat else "No",
            })
confused_pairs_df = pd.DataFrame(confused_pairs).sort_values("Error Count", ascending=False).head(15).reset_index(drop=True)
confused_pairs_df.to_csv(OUT_DIR / "tables" / "top_confused_pairs.csv", index=False)

print("═" * 80)
print(f"TOP CONFUSED INTENT PAIRS — {BEST_MODEL} (Combined Multi-Noise)")
print("═" * 80)
display(confused_pairs_df)

# 2. Categorized Error Breakdown across all 5 Individual Noise Types + Combined
failure_rows = []
for suite_name in ["clean"] + NOISE_TYPES:
    s_df = TEST_SUITES[suite_name]
    res = ALL_RESULTS[BEST_MODEL][suite_name]
    y_t = s_df["label"].values
    y_p = res["y_pred"]
    confs = res["confidences"]
    texts = s_df["text"].tolist()
    clean_orig = TEST_SUITES["clean"]["text"].tolist()

    for idx in range(len(y_t)):
        if y_t[idx] != y_p[idx]:
            t_lbl = ID2LABEL[y_t[idx]]
            p_lbl = ID2LABEL[y_p[idx]]
            same_cat = INTENT_TO_CATEGORY[t_lbl] == INTENT_TO_CATEGORY[p_lbl]
            conf = float(confs[idx])

            if same_cat:
                cause = "Sibling-Intent Semantic Overlap (Same Category)"
            elif conf >= 0.85:
                cause = "High-Confidence Misclassification (Spurious Cue)"
            elif suite_name in ("typos", "abbreviations"):
                cause = "Severe Lexical/Subword Corruption"
            elif suite_name == "indirect":
                cause = "Indirect Hedging / Intent Dilution"
            elif suite_name == "combined":
                cause = "Compounded Multi-Noise Degradation"
            else:
                cause = "Cross-Category Ambiguity"

            failure_rows.append({
                "Noise Suite": suite_name,
                "Original Clean Text": clean_orig[idx],
                "Noisy Input": texts[idx],
                "True Intent": t_lbl,
                "Predicted Intent": p_lbl,
                "Confidence": round(conf, 3),
                "Failure Category": cause,
            })

failures_df = pd.DataFrame(failure_rows)
failures_df.to_csv(OUT_DIR / "tables" / "detailed_failure_cases.csv", index=False)

print("\n" + "═" * 80)
print(f"FAILURE MODE TAXONOMY ({len(failures_df)} total errors across all 7 test suites)")
print("═" * 80)
if len(failures_df) > 0:
    display(failures_df["Failure Category"].value_counts().to_frame("Count"))
    print("\nRepresentative Failure Cases (Sampled Across Noise Types):")
    display(failures_df.groupby("Noise Suite", group_keys=False).apply(lambda g: g.head(3)).reset_index(drop=True))
else:
    print("Zero errors recorded!")

### Root-Cause Findings, Limitations & Threats to Validity

**Where Does the System Fail? (Evidence-Based Summary)**
1. **Sibling-Intent Overlap (Dominant Failure Mode):** Most remaining errors occur between semantically adjacent intents within the same category (e.g., `cancel_order` vs `change_order`, `get_refund` vs `track_refund`, `check_invoice` vs `get_invoice`). When typos or grammatical drops strip a single discriminative verb (*"cancel"* vs *"change"*), the surrounding context is nearly identical.
2. **Compounded Multi-Noise (`combined`):** Chaining indirect conversational templates with heavy typos and abbreviations dilutes intent-bearing tokens and pushes rare subwords into fragmented WordPiece pieces.
3. **High-Confidence Errors:** A small fraction of errors still occur above 0.85 confidence, typically triggered by keyword collisions (e.g., mentioning *"refund"* while actually asking about *"payment_issue"*). Temperature scaling + confidence thresholding mitigates this by escalating borderline queries (`conf < 0.75`) to human agents.

**Limitations & Threats to Validity**
- **Synthetic vs. Organic Noise:** Although our 5 noise generators model realistic keyboard typos, SMS abbreviations, slang, grammar drops, and indirect hedging, real production chat logs may contain domain-specific product jargon, emojis, code-switching, or multi-intent utterances (*"Cancel my order AND change my address"*).
- **Single-Intent Assumption:** Bitext is single-label; real customer messages occasionally contain two intents in one turn.
- **Template Placeholders:** Bitext uses `{{Order Number}}`-style placeholders; in live production, a lightweight NER/regex PII masker should run upstream before intent classification.

## 11. Tested Results Showcase — Real-World Messy Customer Queries

Below we test **14 realistic, hand-crafted customer support messages** (never seen during training) spanning Clean, Typos, Abbreviations, Slang, Grammatical Errors, Indirect Language, and Extreme Combined Noise, comparing **LogReg (Robust)**, **BiLSTM (Robust)**, **DistilBERT (Clean)**, and **DistilBERT (Robust)** side-by-side.

In [ ]:
HANDCRAFTED_QUERIES = [
    ("Clean", "I would like to cancel my recent order right away.", "cancel_order"),
    ("Clean", "How can I reset my forgotten account password?", "recover_password"),
    ("Typos", "pls hlep me trakc my pakcage delivry stauts", "track_order"),
    ("Typos", "i wnat to get a rfuedn for my damged item", "get_refund"),
    ("Abbreviations", "pls cncl my ord asap & send cnfrm msg", "cancel_order"),
    ("Abbreviations", "need 2 upd my ship addr b4 deliv", "change_shipping_address"),
    ("Slang", "yo bro i wanna talk to a real human agent rn fr", "contact_human_agent"),
    ("Slang", "ngl u guys messed up my bill, tryna file a complaint asap", "complaint"),
    ("Grammar", "where invoice my last order need download", "get_invoice"),
    ("Grammar", "me wanting delete account permanent no more use", "delete_account"),
    ("Indirect", "Hey sorry to bother you, I was just wondering if there is any chance I could switch the payment method I used?", "payment_issue"),
    ("Indirect", "My friend told me you guys have a newsletter, not sure where to click to stop getting those emails though", "newsletter_subscription"),
    ("Combined (Extreme)", "yo tbh idk if this right place but pls hlep cncl my ord #4421 rn bc i ordered wrong thing smh", "cancel_order"),
    ("Combined (Extreme)", "hey fam u guys still havent sent my cash back, tryna chk rfnd stat asap fr", "track_refund"),
]

showcase_models = ["LogReg (Robust)", "BiLSTM (Robust)", "DistilBERT (Clean)", "DistilBERT (Robust)"]
q_texts = [q[1] for q in HANDCRAFTED_QUERIES]
model_preds = {m: MODEL_PREDICTORS[m](q_texts) for m in showcase_models}

rows = []
for i, (ntype, query, expected) in enumerate(HANDCRAFTED_QUERIES):
    row = {"Noise Type": ntype, "Customer Query": query, "Expected Intent": expected}
    for m in showcase_models:
        p = model_preds[m][i]
        pred_lbl = ID2LABEL[int(p.argmax())]
        conf = float(p.max())
        mark = "✓" if pred_lbl == expected else "✗"
        row[m] = f"{mark} {pred_lbl} ({conf*100:.1f}%)"
    # Escalation flag for best model
    best_conf = float(model_preds["DistilBERT (Robust)"][i].max())
    row["Action (τ=0.75)"] = "AUTO-ROUTE" if best_conf >= CONFIG["default_confidence_threshold"] else "ESCALATE → HUMAN"
    rows.append(row)

tested_df = pd.DataFrame(rows)
tested_df.to_csv(OUT_DIR / "tables" / "handcrafted_tested_results.csv", index=False)
display(tested_df)

## 12. M5 — Interactive Live Dashboard & Deployment Prototype

### 12.1 In-Notebook Interactive Presentation Dashboard (`ipywidgets`)
Use this widget live in front of the professors to:
- Enter **any custom customer message**
- Optionally inject **any of the 5 noise types** at `Low / Medium / High` severity on the fly
- Adjust the **Human-Escalation Confidence Threshold (`τ`)**
- Inspect **Top-5 Intent Probabilities**, **Latency**, **Routing Decision**, and **Side-by-Side Model Predictions**!

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

query_input = widgets.Textarea(
    value="yo bro pls cncl my ord asap, i ordered the wrong item smh",
    description="Message:",
    layout=widgets.Layout(width="90%", height="75px"),
)
noise_dropdown = widgets.Dropdown(
    options=[("None (Use Text As-Is)", "none")] + [(nt.capitalize(), nt) for nt in NOISE_TYPES],
    value="none",
    description="Add Noise:",
)
sev_dropdown = widgets.Dropdown(
    options=["low", "medium", "high"],
    value="medium",
    description="Severity:",
)
thresh_slider = widgets.FloatSlider(
    value=CONFIG["default_confidence_threshold"],
    min=0.30, max=0.95, step=0.05,
    description="Threshold τ:",
)
run_btn = widgets.Button(description="Classify Intent", button_style="primary", icon="search")
dash_out = widgets.Output()


def on_classify_clicked(_):
    with dash_out:
        clear_output(wait=True)
        raw_text = query_input.value.strip()
        if not raw_text:
            print("Please enter a customer message.")
            return

        if noise_dropdown.value != "none":
            eval_text = NOISE_FN_MAP[noise_dropdown.value](raw_text, severity=sev_dropdown.value, rng=random.Random(42))
            print(f"Original Input: {raw_text}")
            print(f"Noisy Input ({noise_dropdown.value}/{sev_dropdown.value}): {eval_text}\n")
        else:
            eval_text = raw_text
            print(f"Input Message: {eval_text}\n")

        # Run DistilBERT (Robust) with timing
        t0 = time.time()
        probs_best = MODEL_PREDICTORS["DistilBERT (Robust)"]([eval_text])[0]
        lat_ms = (time.time() - t0) * 1000.0
        top_idx = int(probs_best.argmax())
        top_intent = ID2LABEL[top_idx]
        top_cat = INTENT_TO_CATEGORY[top_intent]
        top_conf = float(probs_best[top_idx])
        tau = thresh_slider.value
        decision = "AUTO-ROUTE TO BOT WORKFLOW" if top_conf >= tau else "ESCALATE TO HUMAN AGENT (Low Confidence)"

        print("═" * 70)
        print(f"  FINAL PREDICTION (DistilBERT Robust):  {top_intent}  [Category: {top_cat}]")
        print(f"  CONFIDENCE SCORE:                      {top_conf*100:.2f}%   (Threshold: {tau*100:.0f}%)")
        print(f"  ROUTING DECISION:                      {decision}")
        print(f"  INFERENCE LATENCY:                     {lat_ms:.2f} ms")
        print("═" * 70)

        # Side-by-side comparison across all models
        comp_rows = []
        for m in ["LogReg (Clean)", "LogReg (Robust)", "LinearSVM (Robust)", "BiLSTM (Robust)", "DistilBERT (Clean)", "DistilBERT (Robust)"]:
            p = MODEL_PREDICTORS[m]([eval_text])[0]
            idx = int(p.argmax())
            comp_rows.append({
                "Model": m,
                "Predicted Intent": ID2LABEL[idx],
                "Category": INTENT_TO_CATEGORY[ID2LABEL[idx]],
                "Confidence (%)": round(float(p[idx]) * 100, 2),
            })
        display(pd.DataFrame(comp_rows))

        # Top-5 Probability Bar Chart
        top5_idx = np.argsort(probs_best)[::-1][:5]
        top5_labels = [ID2LABEL[i] for i in top5_idx]
        top5_probs = [probs_best[i] * 100 for i in top5_idx]

        fig, ax = plt.subplots(figsize=(8, 3.2))
        colors = ["#27ae60" if top5_probs[0] >= tau * 100 else "#e67e22"] + ["#95a5a6"] * 4
        ax.barh(top5_labels[::-1], top5_probs[::-1], color=colors[::-1])
        ax.axvline(tau * 100, color="red", linestyle="--", label=f"Escalation Threshold ({tau*100:.0f}%)")
        ax.set_xlim(0, 105)
        ax.set_xlabel("Confidence (%)")
        ax.set_title("Top-5 Predicted Intents — DistilBERT (Robust)", fontweight="bold")
        for i, v in enumerate(top5_probs[::-1]):
            ax.text(v + 1.0, i, f"{v:.1f}%", va="center", fontsize=9)
        ax.legend(loc="lower right")
        plt.tight_layout()
        plt.show()


run_btn.on_click(on_classify_clicked)
controls = widgets.HBox([noise_dropdown, sev_dropdown, thresh_slider, run_btn])
display(widgets.VBox([query_input, controls, dash_out]))

# Trigger once automatically so the notebook renders a live result on "Run All"
on_classify_clicked(None)

### 12.2 Production Monitoring, Failure Handling & AWS SageMaker Deployment Strategy

| Production Pillar | Concrete Implementation Strategy |
|---|---|
| **Failure Handling & Fallback** | Predictions with calibrated confidence `p_max < 0.75` are automatically routed to a **Human Support Agent** or a clarifying follow-up prompt (*"Did you mean cancel an order or change an order?"*). |
| **Monitoring Strategy** | Track daily (1) **Escalation Rate** (`% queries < τ`), (2) **Confidence Distribution Drift (PSI / KS-test)**, (3) **Intent Class Prior Drift**, and (4) **P95 Inference Latency** via Amazon CloudWatch + SageMaker Model Monitor. |
| **Active Learning Loop** | Human-resolved escalated tickets are masked for PII, audited weekly, and fed back into fine-tuning to adapt to emerging slang and product terms. |
| **SageMaker Real-Time Endpoint** | Package `./outputs/models/distilbert_robust_final/` into `model.tar.gz`, upload to S3, and deploy via `sagemaker.huggingface.HuggingFaceModel` on `ml.g4dn.xlarge` (or ONNX-quantized `ml.c6i.large` for low-cost CPU serving). |

In [ ]:
# Optional: Save a complete JSON Executive Summary & (if on SageMaker) show S3 export snippet
summary_report = {
    "generated_at": datetime.now().isoformat(),
    "dataset": CONFIG["dataset_hf_id"],
    "num_classes": NUM_CLASSES,
    "train_size_clean": len(train_df),
    "train_size_phase2_aug": len(train_aug_phase2_df),
    "val_size": len(val_df),
    "test_size": len(test_df),
    "best_model": "DistilBERT (Robust)",
    "best_model_temperature": round(temp_robust, 4),
    "metrics_best_model": {
        suite: {
            "accuracy_pct": round(ALL_RESULTS["DistilBERT (Robust)"][suite]["accuracy"] * 100, 2),
            "macro_f1_pct": round(ALL_RESULTS["DistilBERT (Robust)"][suite]["macro_f1"] * 100, 2),
            "ece": round(ALL_RESULTS["DistilBERT (Robust)"][suite]["ece"], 4),
        }
        for suite in suite_order
    },
    "phase2_vs_phase1_gain_pp": {
        suite: round(
            (ALL_RESULTS["DistilBERT (Robust)"][suite]["accuracy"] - ALL_RESULTS["DistilBERT (Clean)"][suite]["accuracy"]) * 100, 2
        )
        for suite in suite_order
    },
}

with open(OUT_DIR / "reports" / "executive_summary.json", "w") as f:
    json.dump(summary_report, f, indent=2)

print("═" * 80)
print("M6 — FINAL EXECUTIVE SUMMARY (READY FOR PRESENTATION & VIVA)")
print("═" * 80)
print(json.dumps(summary_report, indent=2))
print("\nAll generated artifacts saved under ./outputs/:")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file() and "pytorch_model" not in p.name and "model.safetensors" not in p.name:
        print(f"  - {p}")